<a href="https://colab.research.google.com/github/moiz120/chest-xray-classification/blob/main/chest_xray_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports
import os, cv2, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from skimage.feature import hog
from skimage import exposure

# Dataset config
DATASET_PATH = '/content/drive/MyDrive/CV_Dataset'
CLASSES      = ['NORMAL', 'PNEUMONIA', 'COVID']   # exact folder names
IMG_SIZE     = (224, 224)
SAMPLES_PER_CLASS = 100

# Verify folders exist
for cls in CLASSES:
    path = os.path.join(DATASET_PATH, cls)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {cls}: {count} images found at {path}")

**Q1: HOG + SIFT Feature Extraction**

In [ ]:
# Feature Extraction (HOG + SIFT)

# Data Loading and Preprocessing (Added to fix NameError)
images = []
labels = []

print("Loading images...")
for i, cls in enumerate(CLASSES):
    class_path = os.path.join(DATASET_PATH, cls)
    if not os.path.exists(class_path):
        print(f"Warning: Class path '{class_path}' not found. Skipping.")
        continue

    # Get all image filenames in the class folder
    all_images_in_class = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    # Limit to SAMPLES_PER_CLASS if specified, otherwise load all
    num_samples_to_load = min(len(all_images_in_class), SAMPLES_PER_CLASS) if SAMPLES_PER_CLASS > 0 else len(all_images_in_class)

    print(f"  Loading {num_samples_to_load} images for class '{cls}'...")
    for j in range(num_samples_to_load):
        img_path = os.path.join(class_path, all_images_in_class[j])
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(i)
        else:
            print(f"Warning: Could not read image {img_path}. Skipping.")

images = np.array(images)
labels = np.array(labels)

print(f"Finished loading. Total images: {len(images)}, Total labels: {len(labels)}")
print(f"Image array shape: {images.shape}, Labels array shape: {labels.shape}")

# Proceed with HOG and SIFT feature extraction
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(15, 12))
fig.suptitle("Q1: HOG Visualization & SIFT Keypoints", fontsize=16)

sift = cv2.SIFT_create()

for i, cls in enumerate(CLASSES):
    # Pick one sample from each class
    # Ensure there are images for this class before trying to access
    class_indices = np.where(labels == i)[0]
    if len(class_indices) == 0:
        print(f"No images found for class {cls}. Skipping visualization for this class.")
        continue

    idx = class_indices[0] # Take the first image of this class
    img_bgr = images[idx]
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # HOG
    hog_features, hog_image = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True
    )
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    print(f"[{cls}] HOG feature vector shape: {hog_features.shape}")

    # SIFT
    keypoints, descriptors = sift.detectAndCompute(img_gray, None)
    sift_img = cv2.drawKeypoints(img_rgb.copy(), keypoints, None,
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    print(f"[{cls}] Number of SIFT keypoints: {len(keypoints)}")

    # Plot
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"{cls} — Original")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(hog_image_rescaled, cmap='gray')
    axes[i, 1].set_title(f"{cls} — HOG Visualization")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(sift_img)
    axes[i, 2].set_title(f"{cls} — SIFT Keypoints ({len(keypoints)})")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

**Q2: KNN + SVM Classification**

In [ ]:
# Extract HOG features from all 900 images
def extract_hog_features(images):
    features = []
    for img in images:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), visualize=False)
        features.append(feat)
    return np.array(features)

print("Extracting HOG features (this may take a minute)...")
X = extract_hog_features(images)
y = labels
print(f"Feature matrix shape: {X.shape}")

# Train/Test Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# KNN (k=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"\nKNN Accuracy: {acc_knn:.4f}")
print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn, target_names=CLASSES))

# SVM (RBF kernel)
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"\nSVM Accuracy: {acc_svm:.4f}")
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm, target_names=CLASSES))

# Confusion Matrix for better model
better_pred = y_pred_svm if acc_svm >= acc_knn else y_pred_knn
better_name = "SVM" if acc_svm >= acc_knn else "KNN"

cm = confusion_matrix(y_test, better_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Q2: Confusion Matrix — {better_name} (Better Model)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

**CNN with Keras**

In [ ]:
# CNN / Deep Learning

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

# Prepare data
X_cnn = images.astype('float32') / 255.0      # normalize to [0, 1]
y_cnn = to_categorical(labels, num_classes=3)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cnn, y_cnn, test_size=0.2, random_state=42, stratify=labels)

# Build CNN
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train
history = model.fit(X_train_c, y_train_c,
                    epochs=10, batch_size=32,
                    validation_split=0.1, verbose=1)

# Evaluate
test_loss, test_acc = model.evaluate(X_test_c, y_test_c, verbose=0)
print(f"\nFinal CNN Test Accuracy: {test_acc:.4f}")

# Plot accuracy & loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title("Q3: CNN Accuracy Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title("Q3: CNN Loss Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

**Q4: YOLOv8 Object Detection**

In [ ]:
# Object Detection using YOLOv8

!pip install ultralytics -qq # Install ultralytics library silently

from ultralytics import YOLO
from PIL import Image as PILImage

# Load pretrained YOLOv8n
yolo_model = YOLO('yolov8n.pt')

# Pick 3 sample X-ray images (one from each class)
sample_paths = []
for cls in CLASSES:
    cls_folder = os.path.join(DATASET_PATH, cls)
    fnames = os.listdir(cls_folder)
    sample_paths.append(os.path.join(cls_folder, fnames[0]))

# Run inference & display results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Q4: YOLOv8 Inference on X-Ray Samples", fontsize=16)

for i, (path, cls) in enumerate(zip(sample_paths, CLASSES)):
    results = yolo_model(path, verbose=False)
    result = results[0]
    annotated = result.plot()              # BGR numpy array with boxes drawn
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    # Print detections
    print(f"\n[{cls} image] Detections:")
    for box in result.boxes:
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        cls_name = yolo_model.names[cls_id]
        print(f"  Class: {cls_name}, Confidence: {conf:.2f}")
    if len(result.boxes) == 0:
        print("  No objects detected (YOLOv8 is trained on COCO, not X-rays)")

    axes[i].imshow(annotated_rgb)
    axes[i].set_title(f"{cls} — YOLO Output")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

**Q5: Vision Transformer (ViT)**

In [ ]:
# Vision Transformer (ViT)

from transformers import ViTForImageClassification, ViTImageProcessor # Changed AutoFeatureExtractor to ViTImageProcessor
from PIL import Image as PILImage
import torch

# Load pretrained ViT
model_name = "google/vit-base-patch16-224"
feature_extractor = ViTImageProcessor.from_pretrained(model_name) # Changed AutoFeatureExtractor to ViTImageProcessor
vit_model = ViTForImageClassification.from_pretrained(model_name)
vit_model.eval()

# Run inference on 3 sample images
print("ViT Predictions on 3 X-Ray Samples:\n")
cnn_preds_q5 = []  # store CNN predictions for comparison

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Q5: ViT Inference on X-Ray Samples", fontsize=16)

for i, (path, cls) in enumerate(zip(sample_paths, CLASSES)):
    pil_img = PILImage.open(path).convert("RGB")
    inputs = feature_extractor(images=pil_img, return_tensors="pt")

    with torch.no_grad():
        outputs = vit_model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_idx = probs.argmax(-1).item()
        confidence = probs[0, pred_idx].item()

    pred_label = vit_model.config.id2label[pred_idx]
    print(f"[{cls} image] ViT Predicted: '{pred_label}' | Confidence: {confidence:.4f}")

    # CNN prediction for the same image
    img_arr = cv2.resize(cv2.imread(path), IMG_SIZE).astype('float32') / 255.0
    cnn_input = np.expand_dims(img_arr, axis=0)
    cnn_out = model.predict(cnn_input, verbose=0)
    cnn_pred_cls = CLASSES[np.argmax(cnn_out)]
    cnn_preds_q5.append(cnn_pred_cls)
    print(f"[{cls} image] CNN Predicted: '{cnn_pred_cls}'")

    axes[i].imshow(pil_img)
    axes[i].set_title(f"True: {cls}\nViT: {pred_label[:20]}\nCNN: {cnn_pred_cls}", fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Comparison Summary
print("\n── ViT vs CNN Comparison ──")
print("ViT is a transformer-based model pretrained on ImageNet-21k with global attention,")
print("making it strong at capturing long-range features. CNN relies on local convolutions")
print("and may miss global context. On X-rays, ViT predictions tend to be more semantically")
print("rich but both models are not fine-tuned on medical data here.")